# 11 — Paired SLM-SQL sampling matrix: author CSC vs CSC LoRA

Evaluate **greedy, pass@k, self_consistency@k, author SLM-SQL/CSC@k, LoRA CSC@k, adaptive SC, adaptive + author CSC, adaptive + LoRA CSC**. All generator draws come from notebook **09**. Author/LoRA share identical fixed or adaptive candidate sets, top-2 prompts, base checkpoint, merge seed and sample budget.

Restore the complete experiment directory and config from **09–10** before running. Notebook **12** can then run the validation policy replay and adaptive report stage. Set `CSC_KS` (for example `4,8,16,32,64`) and `CSC_EVAL_SEEDS`; review both before setting `RUN_CSC_FREEZE=1` or `RUN_CSC_GPU=1`. Freeze is skipped when neither flag is enabled. The session uses the **inference** dependency profile.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

configured = os.environ.get('SLM_PRJ_PATH')
candidates = ([Path(configured)] if configured else []) + [Path.cwd(), *Path.cwd().parents, Path('/kaggle/working/SLM-PRJ')]
REPO_DIR = next((p.resolve() for p in candidates if (p / 'research/slm_sql').is_dir()), None)
if REPO_DIR is None:
    REPO_DIR = Path('/kaggle/working/SLM-PRJ')
    subprocess.run(['git', 'clone', os.environ.get('SLM_PRJ_REPO_URL', 'https://github.com/minhadcaz/SLMSQL_project.git'), str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))


In [ ]:
if os.environ.get('INSTALL_CSC_DEPS', '0') == '1':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-r', str(REPO_DIR / 'requirements-csc-inference.txt')], check=True)


In [ ]:
from scripts.csc_notebook import configure, run
CONFIG, cfg = configure(REPO_DIR)
SEED = int(os.environ.get('CSC_SEED', '42'))
RUN_GPU = os.environ.get('RUN_CSC_GPU', '0') == '1'
print({'config': str(CONFIG), 'experiment': cfg['experiment_dir'], 'seed': SEED, 'run_gpu': RUN_GPU})
run(REPO_DIR, CONFIG, 'plan', '--seed', str(SEED))

# Pilot defaults to one inference seed; full runs default to three.
_default_seeds = str(SEED) if cfg['data']['evaluation_limit'] else '42,43,44'
EVAL_SEEDS = [int(s) for s in os.environ.get('CSC_EVAL_SEEDS', _default_seeds).split(',')]
if not EVAL_SEEDS or len(set(EVAL_SEEDS)) != len(EVAL_SEEDS) or any(s < 0 for s in EVAL_SEEDS):
    raise ValueError('CSC_EVAL_SEEDS must contain unique nonnegative inference seeds.')
DISPLAY_SEED = EVAL_SEEDS[0]
print('Declared inference seeds:', EVAL_SEEDS)


In [ ]:
# Restore only after the full experiment directory/config/pair manifests are in place.
if os.environ.get('RESTORE_CSC_ADAPTER', '0') == '1':
    from scripts.csc_notebook import restore_csc_adapter
    print(restore_csc_adapter(cfg))
else:
    print('Adapter restore is off; expecting experiment_dir/adapter/final.')


In [ ]:
# Declare the final K checkpoints before any freeze. CSC_KS overrides the config sweep.
def _parse_ints(name, value):
    try:
        values = [int(item.strip()) for item in value.split(',') if item.strip()]
    except ValueError as exc:
        raise ValueError(f'{name} must be a comma-separated integer list.') from exc
    if not values or len(values) != len(set(values)) or any(value < 1 for value in values):
        raise ValueError(f'{name} must contain unique positive integers.')
    return values

if os.environ.get('CSC_KS'):
    K_VALUES = _parse_ints('CSC_KS', os.environ['CSC_KS'])
elif os.environ.get('CSC_K'):
    K_VALUES = _parse_ints('CSC_K', os.environ['CSC_K'])
else:
    K_VALUES = [cfg['generation']['evaluation_samples']]
n_samples = cfg['generation']['evaluation_samples']
if any(k > n_samples for k in K_VALUES):
    raise ValueError(f'CSC_KS must be <= evaluation_samples ({n_samples}).')
RUN_FREEZE = RUN_GPU or os.environ.get('RUN_CSC_FREEZE', '0') == '1'
training_manifest = Path(cfg['experiment_dir']) / 'adapter/training_manifest.json'
training_ready = training_manifest.is_file() and json.loads(training_manifest.read_text()).get('status') == 'complete'
print({'declared_seeds': EVAL_SEEDS, 'declared_k_values': K_VALUES, 'run_gpu': RUN_GPU, 'run_freeze': RUN_FREEZE})
if training_ready and RUN_FREEZE:
    print('Freezing these seeds/K values and adaptive settings:', EVAL_SEEDS, K_VALUES)
    run(REPO_DIR, CONFIG, 'freeze', '--seeds', *map(str, EVAL_SEEDS), '--ks', *map(str, K_VALUES))
elif training_ready:
    print('Freeze skipped: set RUN_CSC_GPU=1 or RUN_CSC_FREEZE=1 after reviewing the declared seeds/K values.')
if RUN_GPU and training_ready:
    for evaluation_seed in EVAL_SEEDS:
        for k_value in K_VALUES:
            run(REPO_DIR, CONFIG, 'sampling-final', '--seed', str(evaluation_seed), '--k', str(k_value))
else:
    print('Final merger stages pending: complete adapter + RUN_CSC_GPU=1 are required; no generator is run here.')


## CPU scoring and paired multi-seed report

Require every declared method on exactly the same questions. Missing, modified or incompatible artifacts produce an error rather than a fabricated score. Bootstrap averages inference seeds within each question; it never treats seed/question rows as independent observations. Output costs include both generator and merger tokens/samples. Prefix-k generation latency is unavailable when only full-n batch timing exists.


In [ ]:
complete_seeds = []
for evaluation_seed in EVAL_SEEDS:
    stages = [f'{variant}.csc-k{k}' for k in K_VALUES for variant in ('author', 'lora')]
    if cfg['adaptive']['enabled']:
        stages += [f'{variant}.adaptive-csc' for variant in ('author', 'lora')]
    manifests = [Path(cfg['experiment_dir']) / f'seed{evaluation_seed}' / f'evaluation.{stage}.jsonl.manifest.json'
                 for stage in stages]
    if all(p.is_file() and json.loads(p.read_text()).get('complete') for p in manifests):
        for k_value in K_VALUES:
            run(REPO_DIR, CONFIG, 'suite', '--seed', str(evaluation_seed), '--k', str(k_value))
        complete_seeds.append(evaluation_seed)
if complete_seeds == EVAL_SEEDS:
    run(REPO_DIR, CONFIG, 'report-suite', '--seeds', *map(str, EVAL_SEEDS))
else:
    print('Complete paired predictions required for declared seeds:', EVAL_SEEDS)


In [ ]:
for k_value in K_VALUES:
    report_path = Path(cfg['experiment_dir']) / 'reports' / f'evaluation-seed{DISPLAY_SEED}-k{k_value}.sampling.comparison.json'
    if report_path.exists():
        report = json.loads(report_path.read_text())
        print('K =', k_value)
        for row in report['table']:
            print(row)
        print('Paired changes:', report['paired_comparisons'])
        print('CSV table:', report_path.with_suffix('.csv'))
    else:
        print('No complete sampling comparison exists yet:', report_path)

aggregate = Path(cfg['experiment_dir']) / 'reports' / 'evaluation.sampling.comparison.multi-seed.json'
if aggregate.is_file():
    summary = json.loads(aggregate.read_text())['summary']
    for name, metrics in summary.items():
        print(name, {key: metrics[key] for key in ('ex_mean', 'ex_std', 'mean_samples', 'mean_merge_samples', 'mean_tokens')})
